In [6]:
import pandas as pd

# Loads the CSV using its exact path
df = pd.read_csv("/Global Superstore.csv", encoding="latin1")

# Displays the first 5 rows
display(df.head())


,Date,Region,Product,Salesperson,Units_Sold,Unit_Price,Category,Revenue,Cost,Profit
0,2024-04-12 00:00:00,North,Smartwatch,Hannah,15.0,1224.0,Accessories,18360.0,16451.634260,1908.365742
1,2024-12-14 00:00:00,North,Monitor,Eva,5.0,1321.0,Office,6605.0,4457.351727,2147.648273
2,2024-09-27 00:00:00,North,Mobile,Bob,11.0,912.0,Electronics,10032.0,6563.644126,3468.355874
3,2024-04-16 00:00:00,West,Monitor,Charlie,18.0,325.0,Office,5850.0,4320.807092,1529.192908
4,2024-03-12 00:00:00,West,Headphones,Eva,13.0,1042.0,Accessories,13546.0,8270.122666,5275.877334


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import io
import base64

# Ensure df is available
# df = pd.read_csv("/Global Superstore.csv", encoding="latin1")

def query_sales_data(region: str = None, category: str = None, top_n: int = 5):
    """Tool 1: Filters data by region and/or category and returns top items by sales."""
    filtered_df = df.copy()

    if region:
        filtered_df = filtered_df[filtered_df['Region'].str.lower() == region.lower()]
    if category:
        filtered_df = filtered_df[filtered_df['Category'].str.lower() == category.lower()]

    # Group by Product Name or Sub-Category and sum sales
    summary = filtered_df.groupby('Sub-Category')['Sales'].sum().reset_index()
    summary = summary.sort_values(by='Sales', ascending=False).head(top_n)

    return summary.to_dict(orient='records')


def calculate_metrics(metric_type: str = "total_sales", region: str = None):
    """Tool 2: Computes high-level financial metrics or aggregates."""
    filtered_df = df.copy()

    if region:
        filtered_df = filtered_df[filtered_df['Region'].str.lower() == region.lower()]

    if metric_type == "total_sales":
        val = filtered_df['Sales'].sum()
        return {"metric": "Total Sales", "value": round(val, 2)}
    elif metric_type == "total_profit":
        val = filtered_df['Profit'].sum()
        return {"metric": "Total Profit", "value": round(val, 2)}
    elif metric_type == "average_discount":
        val = filtered_df['Discount'].mean()
        return {"metric": "Average Discount", "value": round(val, 4)}

    return {"error": "Unknown metric type"}


def generate_sales_chart(region: str = None):
    """Tool 3: Generates a quick bar chart of sales by Category and returns a confirmation."""
    filtered_df = df.copy()
    if region:
        filtered_df = filtered_df[filtered_df['Region'].str.lower() == region.lower()]

    cat_sales = filtered_df.groupby('Category')['Sales'].sum()

    # Simple plot creation
    plt.figure(figsize=(6, 4))
    cat_sales.plot(kind='bar', color='skyblue')
    plt.title(f"Sales by Category {f'for {region}' if region else ''}")
    plt.ylabel("Sales")
    plt.tight_layout()

    # Save to a text description/status since chatbot text is primary
    plt.close()
    return {"status": "Chart generated successfully", "categories_plotted": list(cat_sales.index)}

print("Tools defined successfully!")

Tools defined successfully!


In [8]:
!pip install -q langgraph langchain langchain-core

In [9]:
import operator
from typing import Annotated, TypedDict, Union, List, Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# 1. Define the Graph State
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    reasoning_trace: str
    tool_to_call: str
    tool_input: dict
    final_answer: str

# 2. Define the Reasoning Node (Where the agent decides what to do)
def reasoning_node(state: AgentState):
    """Analyzes the user's question, writes a reasoning trace, and picks a tool."""
    messages = state["messages"]
    latest_query = messages[-1].content.lower()

    # Simple rule-based reasoning simulation (or you can connect an LLM here)
    if "top" in latest_query or "list" in latest_query or "product" in latest_query or "sub-category" in latest_query:
        reasoning_trace = "The user is asking for specific items or a ranked list. I should use the Data Query Tool."
        tool_to_call = "query_sales_data"
        tool_input = {"top_n": 5}
        if "region" in latest_query:
            # simple keyword check for region
            for reg in ['west', 'east', 'central', 'south']:
                if reg in latest_query:
                    tool_input["region"] = reg.capitalize()
    elif "chart" in latest_query or "plot" in latest_query or "visualize" in latest_query:
        reasoning_trace = "The user wants a visual representation or chart of the categories. I should use the Charting Tool."
        tool_to_call = "generate_sales_chart"
        tool_input = {}
    else:
        reasoning_trace = "The user is asking for a general metric like total sales or profit. I should use the Calculation Tool."
        tool_to_call = "calculate_metrics"
        tool_input = {"metric_type": "total_sales"}
        if "profit" in latest_query:
            tool_input["metric_type"] = "total_profit"

    return {
        "reasoning_trace": reasoning_trace,
        "tool_to_call": tool_to_call,
        "tool_input": tool_input
    }

# 3. Define the Tool Execution Node
def tool_node(state: AgentState):
    """Executes the chosen tool function."""
    tool_name = state["tool_to_call"]
    tool_input = state["tool_input"]

    result = None
    if tool_name == "query_sales_data":
        result = query_sales_data(**tool_input)
    elif tool_name == "calculate_metrics":
        result = calculate_metrics(**tool_input)
    elif tool_name == "generate_sales_chart":
        result = generate_sales_chart(**tool_input)

    return {"messages": [AIMessage(content=str(result))]}

# 4. Define the Response Synthesis Node (Turns raw tool output into a nice analyst answer)
def synthesis_node(state: AgentState):
    """Synthesizes the tool output into a plain-language insight."""
    trace = state["reasoning_trace"]
    last_message = state["messages"][-1].content

    synthesized_response = f"**Reasoning / Plan:** {trace}\n\n**Analyst Insight:** Based on the data analysis, here are the findings: {last_message}"

    return {"final_answer": synthesized_response}

print("Agent nodes defined successfully!")

Agent nodes defined successfully!


In [10]:
# 1. Initialize the State Graph
workflow = StateGraph(AgentState)

# 2. Add Nodes to the Graph
workflow.add_node("reasoning", reasoning_node)
workflow.add_node("tools", tool_node)
workflow.add_node("synthesis", synthesis_node)

# 3. Define the Flow (Edges)
workflow.set_entry_point("reasoning")

# Route directly from reasoning to tools
workflow.add_edge("reasoning", "tools")

# Route from tools to synthesis
workflow.add_edge("tools", "synthesis")

# End the graph after synthesis
workflow.add_edge("synthesis", END)

# 4. Compile the Graph
app_graph = workflow.compile()

print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [12]:
# Test the compiled graph with a sample question
initial_state = {
    "messages": [HumanMessage(content="What are the top products by sales?")]
}

# Invoke the graph
result = app_graph.invoke(initial_state)

print(result["final_answer"])

KeyError: 'Sub-Category'

In [13]:
def query_sales_data(region: str = None, category: str = None, top_n: int = 5):
    """Tool 1: Filters data by region and/or category and returns top items by sales."""
    filtered_df = df.copy()

    if region:
        filtered_df = filtered_df[filtered_df['Region'].str.lower() == region.lower()]
    if category:
        filtered_df = filtered_df[filtered_df['Category'].str.lower() == category.lower()]

    # Using 'Product Name' which is standard in the Superstore dataset
    summary = filtered_df.groupby('Product Name')['Sales'].sum().reset_index()
    summary = summary.sort_values(by='Sales', ascending=False).head(top_n)

    return summary.to_dict(orient='records')

print("Tool updated successfully!")

Tool updated successfully!


In [15]:
# This will print every exact column name in your CSV file
print(df.columns.tolist())

['Date', 'Region', 'Product', 'Salesperson', 'Units_Sold', 'Unit_Price', 'Category', 'Revenue', 'Cost', 'Profit']


In [17]:
def query_sales_data(region: str = None, category: str = None, top_n: int = 5):
    """Tool 1: Filters data by region and/or category and returns top products by revenue."""
    filtered_df = df.copy()

    if region:
        filtered_df = filtered_df[filtered_df['Region'].str.lower() == region.lower()]
    if category:
        filtered_df = filtered_df[filtered_df['Category'].str.lower() == category.lower()]

    # Using your exact column names: 'Product' and 'Revenue'
    summary = filtered_df.groupby('Product')['Revenue'].sum().reset_index()
    summary = summary.sort_values(by='Revenue', ascending=False).head(top_n)

    return summary.to_dict(orient='records')


def calculate_metrics(metric_type: str = "total_revenue", region: str = None):
    """Tool 2: Computes high-level financial metrics using your dataset columns."""
    filtered_df = df.copy()

    if region:
        filtered_df = filtered_df[filtered_df['Region'].str.lower() == region.lower()]

    if metric_type in ["total_revenue", "total_sales"]:
        val = filtered_df['Revenue'].sum()
        return {"metric": "Total Revenue", "value": round(val, 2)}
    elif metric_type == "total_profit":
        val = filtered_df['Profit'].sum()
        return {"metric": "Total Profit", "value": round(val, 2)}
    elif metric_type == "total_cost":
        val = filtered_df['Cost'].sum()
        return {"metric": "Total Cost", "value": round(val, 2)}

    return {"error": "Unknown metric type"}

print("Tools updated with correct column names!")

Tools updated with correct column names!


In [18]:
initial_state = {
    "messages": [HumanMessage(content="What are the top products by sales?")]
}
result = app_graph.invoke(initial_state)
print(result["final_answer"])

**Reasoning / Plan:** The user is asking for specific items or a ranked list. I should use the Data Query Tool.

**Analyst Insight:** Based on the data analysis, here are the findings: [{'Product': 'Tablet', 'Revenue': 2915878.0}, {'Product': 'Smartwatch', 'Revenue': 2768774.0}, {'Product': 'Laptop', 'Revenue': 2679146.0}, {'Product': 'Camera', 'Revenue': 2554986.0}, {'Product': 'Headphones', 'Revenue': 2427325.0}]


In [19]:
%%writefile app.py
import streamlit as st
import pandas as pd
import operator
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# --- 1. Load Data ---
@st.cache_data
def load_data():
    df = pd.read_csv("Global Superstore.csv", encoding="latin1")
    return df

df = load_data()

# --- 2. Define Tools ---
def query_sales_data(region: str = None, category: str = None, top_n: int = 5):
    filtered_df = df.copy()
    if region:
        filtered_df = filtered_df[filtered_df['Region'].str.lower() == region.lower()]
    if category:
        filtered_df = filtered_df[filtered_df['Category'].str.lower() == category.lower()]

    summary = filtered_df.groupby('Product')['Revenue'].sum().reset_index()
    summary = summary.sort_values(by='Revenue', ascending=False).head(top_n)
    return summary.to_dict(orient='records')

def calculate_metrics(metric_type: str = "total_revenue", region: str = None):
    filtered_df = df.copy()
    if region:
        filtered_df = filtered_df[filtered_df['Region'].str.lower() == region.lower()]

    if metric_type in ["total_revenue", "total_sales"]:
        val = filtered_df['Revenue'].sum()
        return {"metric": "Total Revenue", "value": round(val, 2)}
    elif metric_type == "total_profit":
        val = filtered_df['Profit'].sum()
        return {"metric": "Total Profit", "value": round(val, 2)}
    elif metric_type == "total_cost":
        val = filtered_df['Cost'].sum()
        return {"metric": "Total Cost", "value": round(val, 2)}
    return {"error": "Unknown metric type"}

# --- 3. Define LangGraph State & Nodes ---
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    reasoning_trace: str
    tool_to_call: str
    tool_input: dict
    final_answer: str

def reasoning_node(state: AgentState):
    latest_query = state["messages"][-1].content.lower()
    if "top" in latest_query or "list" in latest_query or "product" in latest_query:
        reasoning_trace = "Identified user intent as a ranked product query. Executing Data Query Tool."
        tool_to_call = "query_sales_data"
        tool_input = {"top_n": 5}
    else:
        reasoning_trace = "Identified user intent as a summary financial metric request. Executing Calculation Tool."
        tool_to_call = "calculate_metrics"
        tool_input = {"metric_type": "total_profit" if "profit" in latest_query else "total_revenue"}

    return {"reasoning_trace": reasoning_trace, "tool_to_call": tool_to_call, "tool_input": tool_input}

def tool_node(state: AgentState):
    tool_name = state["tool_to_call"]
    tool_input = state["tool_input"]
    result = query_sales_data(**tool_input) if tool_name == "query_sales_data" else calculate_metrics(**tool_input)
    return {"messages": [AIMessage(content=str(result))]}

def synthesis_node(state: AgentState):
    trace = state["reasoning_trace"]
    last_message = state["messages"][-1].content
    synthesized_response = f"**Agent Reasoning:** {trace}\n\n**Insight Result:** {last_message}"
    return {"final_answer": synthesized_response}

# Build and Compile Graph
workflow = StateGraph(AgentState)
workflow.add_node("reasoning", reasoning_node)
workflow.add_node("tools", tool_node)
workflow.add_node("synthesis", synthesis_node)
workflow.set_entry_point("reasoning")
workflow.add_edge("reasoning", "tools")
workflow.add_edge("tools", "synthesis")
workflow.add_edge("synthesis", END)
app_graph = workflow.compile()

# --- 4. Streamlit UI Layout ---
st.set_page_config(page_title="Insight Copilot - Global Superstore", layout="centered")
st.title("📊 Global Superstore Insight Copilot")
st.markdown("Ask questions about your sales data powered by **LangGraph** and **Pandas**.")

user_query = st.text_input("What would you like to know about your sales data?", "What are the top products by sales?")

if st.button("Generate Insight"):
    if user_query:
        with st.spinner("Analyzing data through LangGraph workflow..."):
            state = {"messages": [HumanMessage(content=user_query)]}
            result = app_graph.invoke(state)
            st.markdown("### Results")
            st.markdown(result["final_answer"])
    else:
        st.warning("Please enter a question.")

Writing app.py


In [25]:
import gradio as gr
import pandas as pd
from langchain_core.messages import HumanMessage

# Define the prediction function that feeds questions directly into your LangGraph app
def run_insight_copilot(user_query):
    if not user_query.strip():
        return "Please enter a valid question."

    # Run through the compiled LangGraph workflow
    state = {"messages": [HumanMessage(content=user_query)]}
    result = app_graph.invoke(state)
    return result["final_answer"]

# Launch the Gradio Interface
demo = gr.Interface(
    fn=run_insight_copilot,
    inputs=gr.Textbox(
        lines=2,
        placeholder="e.g., What are the top products by sales?",
        label="Ask Insight Copilot"
    ),
    outputs=gr.Markdown(label="Agent Response"),
    title="📊 Global Superstore Insight Copilot",
    description="Powered by LangGraph & Pandas"
)

demo.launch(share=True, inline=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f134982691f6f8dfa7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
